# 03 - Data Preprocessing & Feature Engineering
## RailOpt AI: AI-Powered Railway Maintenance Optimization System
### Smart India Hackathon

---

## Section 1: Introduction

### The Crucial Role of Data Preprocessing in Railway Maintenance Prediction
In industrial AI systems for railway infrastructure, data preprocessing is the foundational phase that bridges raw, noisy operational telemetry and production-grade machine learning models. High-stakes railway decisions—such as issuing emergency speed restrictions or halting traffic along a high-density corridor for track renewal—demand rigorous data hygiene, mathematical consistency, and domain-grounded feature signals.

### Why Preprocessing is Necessary Before Model Training
1. **Data Integrity & Sanitation:** Raw sensor feeds and field inspection logs frequently contain missing measurements, transmission timeouts, duplicate timestamps, and physical anomalies. Validating domain boundaries prevents biased optimization.
2. **Mathematical Representation:** Machine learning models (e.g. gradient boosted trees, regularized linear estimators) operate strictly on numerical matrices. Textual attributes such as asset classes (`Track`, `Signal`, `Bridge`), weather states, and defect severities must be systematically encoded.
3. **Variance Harmonization (Feature Scaling):** Raw attributes span disparate physical dimensions—from temperatures in °C to annual train passages exceeding 40,000 and axle loads in tonnes. Standardizing continuous variables ensures balanced gradient descent convergence and fair feature weighting.
4. **Domain Signal Amplification (Feature Engineering):** Raw features alone rarely capture complex physical degradation. Synthesizing composite domain indicators (e.g. track traffic burden, defect pressure ratios, maintenance overdue indexes) directly injects physical railway engineering principles into the model.
5. **Data Leakage Prevention:** Establishing strict boundaries between Train (70%), Validation (15%), and Test (15%) partitions ensures scalers and encoders are fitted solely on training observations and applied out-of-sample.

> **Important:** In adherence to clean pipeline architecture, this notebook focuses strictly on data preparation. **No machine learning models are trained in this notebook.**

---
## Section 2: Import Libraries

### Purpose
Import standard data science, modeling utilities, and persistence packages required for data manipulation, categorical encoding, feature scaling, and reproducible dataset partitioning.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Resolve the project root whether the notebook runs from the repo or notebooks/.
current_dir = Path.cwd()
PROJECT_ROOT = current_dir if (current_dir / "data").exists() else current_dir.parent
RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "railway_synthetic_training_dataset_7000.xlsx"
)
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
ENCODERS_DIR = PROJECT_ROOT / "outputs" / "models" / "encoders"
MODELS_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "models"

for output_dir in (PROCESSED_DATA_DIR, ENCODERS_DIR, MODELS_OUTPUT_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source workbook found: {RAW_DATA_PATH.exists()}")
print(f"Processed data directory: {PROCESSED_DATA_DIR}")
print(f"Encoder directory: {ENCODERS_DIR}")
print(f"Model artifact directory: {MODELS_OUTPUT_DIR}")

Project root: c:\Users\DAKSHITHA S\Downloads\Block - train
Source workbook found: True
Processed data directory: c:\Users\DAKSHITHA S\Downloads\Block - train\data\processed
Encoder directory: c:\Users\DAKSHITHA S\Downloads\Block - train\outputs\models\encoders
Model artifact directory: c:\Users\DAKSHITHA S\Downloads\Block - train\outputs\models


---
## Section 3: Load Dataset

### Purpose
Load the immutable raw railway maintenance dataset from `data/raw/` using `pathlib.Path` and inspect its initial shape and schema attributes.

In [2]:
assert RAW_DATA_PATH.exists(), f"Source dataset not found at {RAW_DATA_PATH}"

# Read raw Excel dataset
df_raw = pd.read_excel(RAW_DATA_PATH)

# Make an explicit working copy for preprocessing
df = df_raw.copy()

print("=" * 50)
print("DATASET INGESTION VERIFICATION")
print("=" * 50)
print(f"File Name     : {RAW_DATA_PATH.name}")
print(f"Initial Shape : {df.shape[0]:,} Rows | {df.shape[1]} Columns")
print("=" * 50)
df.head(3)

DATASET INGESTION VERIFICATION
File Name     : railway_synthetic_training_dataset_7000.xlsx
Initial Shape : 7,000 Rows | 22 Columns


,asset_id,asset_type,zone,station,asset_age_years,last_maintenance_days,defect_count,defect_severity,inspection_score,train_frequency,...,temperature,rainfall,axle_load,crew_available,maintenance_duration,passenger_peak,nearby_alternate_track,previous_failures,predicted_failure_probability,priority_score
0,AST00001,OHE,West,STN_005,25,676,8,High,26.0,332,...,41.6,131.7,26.4,5,10,No,No,5,0.990,92
1,AST00002,Bridge,South,STN_048,25,48,5,Medium,45.0,79,...,42.3,147.4,21.4,20,12,No,No,2,0.878,78
2,AST00003,Switch,East,STN_021,39,122,3,Medium,50.8,287,...,28.8,43.5,31.6,3,6,Yes,No,7,0.990,87


---
## Section 4: Missing Value Treatment

### Strategy & Implementation
- **Numerical Columns:** Impute with column **median** (robust to extreme values and skewness).
- **Categorical Columns:** Impute with column **mode** (most frequent category).
- Generate a comprehensive imputation audit report showing detected null counts and replacement values applied.

In [3]:
def treat_missing_values(data: pd.DataFrame) -> pd.DataFrame:
    """Impute numeric columns with medians and categorical columns with modes."""
    cleaned = data.copy()
    missing_summary = []

    for column in cleaned.columns:
        missing_count = int(cleaned[column].isna().sum())
        if missing_count == 0:
            continue

        if pd.api.types.is_numeric_dtype(cleaned[column]):
            fill_value = cleaned[column].median()
            strategy = "median"
        else:
            modes = cleaned[column].mode(dropna=True)
            if modes.empty:
                raise ValueError(
                    f"Cannot impute '{column}': the column contains no observed values."
                )
            fill_value = modes.iloc[0]
            strategy = "mode"

        if pd.isna(fill_value):
            raise ValueError(
                f"Cannot impute '{column}': no valid {strategy} could be calculated."
            )

        cleaned[column] = cleaned[column].fillna(fill_value)
        missing_summary.append(
            {
                "Feature": column,
                "Type": "Numeric" if strategy == "median" else "Categorical",
                "Missing Fixed": missing_count,
                "Strategy": strategy,
                "Fill Value": fill_value,
            }
        )

    global missing_values_fixed
    missing_values_fixed = sum(item["Missing Fixed"] for item in missing_summary)
    print(f"Missing cells fixed: {missing_values_fixed:,}")
    if missing_summary:
        display(pd.DataFrame(missing_summary))
    else:
        print("No missing values were found.")

    return cleaned


df = treat_missing_values(df)

Missing cells fixed: 0
No missing values were found.


---
## Section 5: Duplicate Removal

### Rationale
Duplicate observation rows represent redundant data logging that can artificially inflate validation metrics, induce overfitting, and distort tree node split thresholds.

In [4]:
rows_before = len(df)
duplicate_count = df.duplicated().sum()

# Remove duplicates if any exist
df = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df)
rows_removed = rows_before - rows_after

print("=" * 45)
print("DUPLICATE OBSERVATION REMOVAL AUDIT")
print("=" * 45)
print(f"Rows Before Deduplication : {rows_before:,}")
print(f"Rows After Deduplication  : {rows_after:,}")
print(f"Number of Rows Removed    : {rows_removed}")
print("=" * 45)
if rows_removed == 0:
    print("✅ Verification Passed: Entire dataset consists of unique asset records.")

DUPLICATE OBSERVATION REMOVAL AUDIT
Rows Before Deduplication : 7,000
Rows After Deduplication  : 7,000
Number of Rows Removed    : 0
✅ Verification Passed: Entire dataset consists of unique asset records.


---
## Section 6: Data Validation & Domain Sanity Checks

### Enforced Physical Domain Constraints
1. **`priority_score`:** Must strictly reside between **0 and 100**.
2. **`predicted_failure_probability`:** Must strictly reside between **0.0 and 1.0**.
3. **`inspection_score`:** Must strictly reside between **0.0 and 100.0**.
4. **Non-Negative Attributes:** Track age, days since maintenance, defect counts, train speed, train frequency, axle load, and crew availability must be $\ge 0$.

Invalid observations violating fundamental physical principles are identified and pruned.

In [5]:
def validate_railway_data(data: pd.DataFrame) -> pd.DataFrame:
    """Remove rows that violate target bounds or physical domain constraints."""
    required_columns = [
        "priority_score",
        "predicted_failure_probability",
        "inspection_score",
        "asset_age_years",
        "last_maintenance_days",
        "defect_count",
        "train_frequency",
        "average_speed",
        "axle_load",
        "crew_available",
        "maintenance_duration",
        "rainfall",
        "previous_failures",
    ]
    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise KeyError(f"Required columns are missing: {missing_columns}")

    validated = data.copy()
    initial_count = len(validated)
    numeric_columns = validated.select_dtypes(include=[np.number]).columns
    finite_values = np.isfinite(validated[numeric_columns]).all(axis=1)

    valid_priority = validated["priority_score"].between(0, 100)
    valid_probability = validated["predicted_failure_probability"].between(0, 1)
    valid_inspection = validated["inspection_score"].between(0, 100)

    non_negative_columns = [
        "asset_age_years",
        "last_maintenance_days",
        "defect_count",
        "train_frequency",
        "average_speed",
        "axle_load",
        "crew_available",
        "maintenance_duration",
        "rainfall",
        "previous_failures",
    ]
    valid_non_negative = validated[non_negative_columns].ge(0).all(axis=1)
    valid_rows = (
        finite_values
        & valid_priority
        & valid_probability
        & valid_inspection
        & valid_non_negative
    )

    violations = {
        "Non-finite numeric values": int((~finite_values).sum()),
        "priority_score outside [0, 100]": int((~valid_priority).sum()),
        "predicted_failure_probability outside [0, 1]": int(
            (~valid_probability).sum()
        ),
        "inspection_score outside [0, 100]": int((~valid_inspection).sum()),
        "Negative physical values": int((~valid_non_negative).sum()),
    }
    validated = validated.loc[valid_rows].reset_index(drop=True)
    global invalid_rows_removed
    invalid_rows_removed = initial_count - len(validated)

    print(f"Rows before validation: {initial_count:,}")
    for check_name, count in violations.items():
        print(f"{check_name}: {count:,}")
    print(f"Invalid rows removed: {invalid_rows_removed:,}")
    print(f"Rows after validation: {len(validated):,}")
    return validated


df = validate_railway_data(df)

Rows before validation: 7,000
Non-finite numeric values: 0
priority_score outside [0, 100]: 0
predicted_failure_probability outside [0, 1]: 0
inspection_score outside [0, 100]: 0
Negative physical values: 0
Invalid rows removed: 0
Rows after validation: 7,000


---
## Section 7: Categorical Encoding

### Rationale & Encoding Scheme
To prepare metadata for machine learning algorithms, the following 6 categorical attributes are transformed using **`LabelEncoder`**:
1. `asset_type` (Track, Signal, Switch, Bridge, OHE)
2. `zone` (North, South, East, West, Central)
3. `weather` (Sunny, Cloudy, Rain, Fog)
4. `defect_severity` (Low, Medium, High)
5. `passenger_peak` (No, Yes)
6. `nearby_alternate_track` (No, Yes)

Every fitted `LabelEncoder` object is serialized and saved using `joblib` into `outputs/models/encoders/` to allow deterministic inverse transformation and production serving.

In [6]:
cat_encode_cols = [
    "asset_type",
    "zone",
    "weather",
    "defect_severity",
    "passenger_peak",
    "nearby_alternate_track",
]
missing_categories = sorted(set(cat_encode_cols) - set(df.columns))
if missing_categories:
    raise KeyError(f"Categorical columns are missing: {missing_categories}")

# Reserve deterministic row partitions so encoders learn categories from train only.
train_indices, holdout_indices = train_test_split(
    df.index.to_numpy(), test_size=0.30, random_state=42, shuffle=True
)
validation_indices, test_indices = train_test_split(
    holdout_indices, test_size=0.50, random_state=42, shuffle=True
)

encoders_saved = {}
unknown_category = "__UNKNOWN__"

for column in cat_encode_cols:
    encoder = LabelEncoder()
    train_values = df.loc[train_indices, column].astype(str)
    encoder.fit(train_values)
    encoder.classes_ = np.unique(
        np.append(encoder.classes_, unknown_category)
    )

    values = df[column].astype(str)
    values = values.where(values.isin(encoder.classes_), unknown_category)
    df[column] = encoder.transform(values)

    encoder_path = ENCODERS_DIR / f"encoder_{column}.joblib"
    joblib.dump(encoder, encoder_path)
    encoders_saved[column] = encoder_path

    mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
    print(f"{column}: {mapping}")
    print(f"Saved encoder: {encoder_path}")

print(f"\nSaved {len(encoders_saved)} train-fitted encoders to {ENCODERS_DIR}")

asset_type: {'Bridge': np.int64(0), 'OHE': np.int64(1), 'Signal': np.int64(2), 'Switch': np.int64(3), 'Track': np.int64(4), '__UNKNOWN__': np.int64(5)}
Saved encoder: c:\Users\DAKSHITHA S\Downloads\Block - train\outputs\models\encoders\encoder_asset_type.joblib
zone: {'Central': np.int64(0), 'East': np.int64(1), 'North': np.int64(2), 'South': np.int64(3), 'West': np.int64(4), '__UNKNOWN__': np.int64(5)}
Saved encoder: c:\Users\DAKSHITHA S\Downloads\Block - train\outputs\models\encoders\encoder_zone.joblib
weather: {'Cloudy': np.int64(0), 'Fog': np.int64(1), 'Rain': np.int64(2), 'Sunny': np.int64(3), '__UNKNOWN__': np.int64(4)}
Saved encoder: c:\Users\DAKSHITHA S\Downloads\Block - train\outputs\models\encoders\encoder_weather.joblib
defect_severity: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2), '__UNKNOWN__': np.int64(3)}
Saved encoder: c:\Users\DAKSHITHA S\Downloads\Block - train\outputs\models\encoders\encoder_defect_severity.joblib
passenger_peak: {'No': np.int64(0

---
## Section 8: Feature Engineering

Create domain features that capture maintenance history, failure risk, traffic load, asset condition, and utilization. Original dataset columns remain in the working dataframe.

- `maintenance_overdue_ratio` = `last_maintenance_days / max(asset_age_years * 365, 1)`.
- `risk_index` = `defect_count * predicted_failure_probability`.
- `traffic_load_score` = `train_frequency * axle_load`.
- `asset_health_score` = `inspection_score - defect_count * 5`.
- `asset_utilization` = `train_frequency / max(average_speed, 1)`.

The 1-unit denominator floors prevent division by zero for new assets or zero recorded speed.

In [7]:
required_feature_columns = [
    "asset_age_years",
    "last_maintenance_days",
    "defect_count",
    "predicted_failure_probability",
    "train_frequency",
    "axle_load",
    "inspection_score",
    "average_speed",
]
missing_feature_columns = sorted(set(required_feature_columns) - set(df.columns))
if missing_feature_columns:
    raise KeyError(f"Cannot engineer features; columns missing: {missing_feature_columns}")

# Protect the ratios from division by zero while preserving the requested formulas.
asset_age_days = (df["asset_age_years"] * 365).clip(lower=1)
safe_speed = df["average_speed"].clip(lower=1)

df["maintenance_overdue_ratio"] = (
    df["last_maintenance_days"] / asset_age_days
)
df["risk_index"] = (
    df["defect_count"] * df["predicted_failure_probability"]
)
df["traffic_load_score"] = df["train_frequency"] * df["axle_load"]
df["asset_health_score"] = (
    df["inspection_score"] - df["defect_count"] * 5
)
df["asset_utilization"] = df["train_frequency"] / safe_speed

engineered_cols = [
    "maintenance_overdue_ratio",
    "risk_index",
    "traffic_load_score",
    "asset_health_score",
    "asset_utilization",
]
print(f"Engineered {len(engineered_cols)} features; original columns retained.")
display(df[engineered_cols].describe().T)
if "asset_id" in df.columns:
    display(df[["asset_id"] + engineered_cols].head(3))

Engineered 5 features; original columns retained.


,count,mean,std,min,25%,50%,75%,max
maintenance_overdue_ratio,7000.0,0.118065,0.228829,0.000125,0.026926,0.053958,0.105864,2.191781
risk_index,7000.0,4.889179,3.167538,0.000000,1.980000,4.950000,7.920000,9.900000
traffic_load_score,7000.0,4631.685900,2675.423639,318.000000,2414.350000,4377.800000,6512.850000,12006.200000
asset_health_score,7000.0,23.721571,39.324782,-50.000000,-9.800000,23.800000,56.800000,100.000000
asset_utilization,7000.0,2.875329,2.006863,0.183486,1.370690,2.479233,3.811233,11.566667


,asset_id,maintenance_overdue_ratio,risk_index,traffic_load_score,asset_health_score,asset_utilization
0,AST00001,0.074082,7.92,8764.8,-14.0,6.036364
1,AST00002,0.005260,4.39,1690.6,20.0,1.012821
2,AST00003,0.008570,2.97,9069.2,35.8,2.813725


---
## Section 9: Outlier Detection

### IQR Audit and Treatment Decision
For each continuous numeric feature, calculate the IQR fences (`Q1 - 1.5 × IQR` and `Q3 + 1.5 × IQR`) and report the number and percentage of observations outside them. Exclude the target, identifiers, and label-encoded categories from this audit because they are not continuous measurements.

No outliers are removed or capped automatically. Railway extremes may represent genuine high-risk conditions; without evidence of a measurement error or a domain-approved physical limit, retaining them is safer and preserves valuable training examples. The report records zero outliers treated.

In [8]:
excluded_from_outlier_audit = set(cat_encode_cols) | {"asset_id", "priority_score"}
numerical_audit_cols = [
    column
    for column in df.select_dtypes(include=[np.number]).columns
    if column not in excluded_from_outlier_audit
]

outlier_audit = []
for column in numerical_audit_cols:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    is_outlier = (df[column] < lower_bound) | (df[column] > upper_bound)
    count = int(is_outlier.sum())

    outlier_audit.append(
        {
            "Feature": column,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Outlier Count": count,
            "Outlier (%)": 100 * count / len(df),
        }
    )

outlier_df = pd.DataFrame(outlier_audit).sort_values(
    by="Outlier Count", ascending=False
)
outliers_treated = 0
print("IQR outlier audit (no automatic removal or capping):")
display(outlier_df.round(2))
print(f"Outliers treated: {outliers_treated}")

IQR outlier audit (no automatic removal or capping):


,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count,Outlier (%)
12,predicted_failure_probability,0.99,0.99,0.00,0.99,0.99,1349,19.27
13,maintenance_overdue_ratio,0.03,0.11,0.08,-0.09,0.22,759,10.84
17,asset_utilization,1.37,3.81,2.44,-2.29,7.47,259,3.70
0,asset_age_years,11.00,31.00,20.00,-19.00,61.00,0,0.00
1,last_maintenance_days,201.00,603.00,402.00,-402.00,1206.00,0,0.00
2,defect_count,2.00,8.00,6.00,-7.00,17.00,0,0.00
6,temperature,18.50,36.00,17.50,-7.75,62.25,0,0.00
3,inspection_score,29.20,68.00,38.80,-29.00,126.20,0,0.00
4,train_frequency,103.00,269.00,166.00,-146.00,518.00,0,0.00
5,average_speed,52.00,97.00,45.00,-15.50,164.50,0,0.00


Outliers treated: 0


---
## Section 10: Train, Validation, and Test Split

Reserve a reproducible 70% training set and two equal 15% holdouts with `random_state=42`. The row indices were established before categorical encoding so encoders can be fitted on training rows only. The split is not stratified because `priority_score` is a continuous regression target.

In [9]:
TARGET_COL = "priority_score"
METADATA_COLUMNS = [
    column for column in ["asset_id", "station"] if column in df.columns
]
X = df.drop(columns=[TARGET_COL, *METADATA_COLUMNS])
y = df[TARGET_COL]

# Use the deterministic row partitions reserved before fitting encoders.
X_train = X.loc[train_indices].copy()
X_val = X.loc[validation_indices].copy()
X_test = X.loc[test_indices].copy()
y_train = y.loc[train_indices].copy()
y_val = y.loc[validation_indices].copy()
y_test = y.loc[test_indices].copy()

print("Dataset partition sizes (70% / 15% / 15%):")
for split_name, split_features in (
    ("Train", X_train),
    ("Validation", X_val),
    ("Test", X_test),
):
    print(
        f"{split_name:<10}: {len(split_features):,} rows "
        f"({len(split_features) / len(df) * 100:.1f}%)"
    )
print(f"Total rows: {len(X_train) + len(X_val) + len(X_test):,}")
print(f"Identifier metadata excluded from model inputs: {METADATA_COLUMNS}")

Dataset partition sizes (70% / 15% / 15%):
Train     : 4,900 rows (70.0%)
Validation: 1,050 rows (15.0%)
Test      : 1,050 rows (15.0%)
Total rows: 7,000
Identifier metadata excluded from model inputs: ['asset_id', 'station']


---
## Section 11: Feature Scaling

Fit `StandardScaler` only on continuous numeric training features, then use it to transform train, validation, and test data. Do not scale `priority_score` or the label-encoded categorical columns; scaling ordinal codes would imply misleading numeric distances. Save the fitted scaler to `outputs/models/scaler.pkl`.

In [11]:
non_scale_cols = set(cat_encode_cols)
scale_features = [
    column
    for column in X_train.select_dtypes(include=[np.number]).columns
    if column not in non_scale_cols
]
if not scale_features:
    raise ValueError("No continuous numeric features are available to scale.")

scaler = StandardScaler()
scaler.fit(X_train[scale_features])


def scale_feature_frame(features: pd.DataFrame) -> pd.DataFrame:
    """Scale selected columns and preserve all other columns and their order."""
    scaled_values = pd.DataFrame(
        scaler.transform(features[scale_features]),
        index=features.index,
        columns=scale_features,
    )
    unscaled_features = features.drop(columns=scale_features)
    return pd.concat([unscaled_features, scaled_values], axis=1).reindex(
        columns=features.columns
    )


X_train_scaled = scale_feature_frame(X_train)
X_val_scaled = scale_feature_frame(X_val)
X_test_scaled = scale_feature_frame(X_test)

scaler_path = MODELS_OUTPUT_DIR / "scaler.pkl"
joblib.dump(scaler, scaler_path)

print(f"Scaled continuous features: {len(scale_features)}")
print(f"Unscaled target: {TARGET_COL}")
print(f"Unscaled encoded categories: {cat_encode_cols}")
print(f"Saved scaler: {scaler_path}")

Scaled continuous features: 18
Unscaled target: priority_score
Unscaled encoded categories: ['asset_type', 'zone', 'weather', 'defect_severity', 'passenger_peak', 'nearby_alternate_track']
Saved scaler: c:\Users\DAKSHITHA S\Downloads\Block - train\outputs\models\scaler.pkl


---
## Section 12: Save Processed Data

Combine each scaled feature split with its original, unscaled `priority_score` and save `train.csv`, `validation.csv`, and `test.csv` to `data/processed/`.

In [12]:
train_processed = X_train_scaled.copy()
train_processed[TARGET_COL] = y_train.to_numpy()

validation_processed = X_val_scaled.copy()
validation_processed[TARGET_COL] = y_val.to_numpy()

test_processed = X_test_scaled.copy()
test_processed[TARGET_COL] = y_test.to_numpy()

processed_splits = {
    "train.csv": train_processed,
    "validation.csv": validation_processed,
    "test.csv": test_processed,
}
files_saved = []
for file_name, split_data in processed_splits.items():
    output_path = PROCESSED_DATA_DIR / file_name
    split_data.to_csv(output_path, index=False)
    files_saved.append(output_path)
    print(f"Saved {output_path.name}: {split_data.shape}")

print(f"Processed CSV files saved in: {PROCESSED_DATA_DIR}")

Saved train.csv: (4900, 25)
Saved validation.csv: (1050, 25)
Saved test.csv: (1050, 25)
Processed CSV files saved in: c:\Users\DAKSHITHA S\Downloads\Block - train\data\processed


---
## Section 13: Preprocessing Audit Report

### End-to-End Pipeline Summary

In [13]:
summary_data = {
    "Pipeline Audit Metric": [
        "Original Dataset Shape",
        "Final Dataset Shape",
        "Features Added",
        "Features Encoded",
        "Features Scaled",
        "Features Dropped from Model Inputs",
        "Missing Values Fixed",
        "Duplicates Removed",
        "Outliers Treated",
        "Train Rows",
        "Validation Rows",
        "Test Rows",
        "Files Saved",
    ],
    "Value": [
        f"{df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns",
        f"{df.shape[0]:,} rows x {df.shape[1]} columns",
        ", ".join(engineered_cols),
        ", ".join(cat_encode_cols),
        f"{len(scale_features)} continuous numeric features; target not scaled",
        ", ".join(METADATA_COLUMNS) if METADATA_COLUMNS else "None",
        f"{missing_values_fixed:,}",
        f"{rows_removed:,}",
        f"{outliers_treated:,} (none removed or capped)",
        f"{len(train_processed):,} ({len(train_processed) / len(df):.1%})",
        f"{len(validation_processed):,} ({len(validation_processed) / len(df):.1%})",
        f"{len(test_processed):,} ({len(test_processed) / len(df):.1%})",
        ", ".join(path.name for path in files_saved)
        + ", scaler.pkl, 6 encoder joblib files",
    ],
}

summary_df = pd.DataFrame(summary_data)
display(summary_df)
print("Preprocessing complete. No machine learning model was trained.")

,Pipeline Audit Metric,Value
0,Original Dataset Shape,"7,000 rows x 22 columns"
1,Final Dataset Shape,"7,000 rows x 27 columns"
2,Features Added,"maintenance_overdue_ratio, risk_index, traffic..."
3,Features Encoded,"asset_type, zone, weather, defect_severity, pa..."
4,Features Scaled,18 continuous numeric features; target not scaled
5,Features Dropped from Model Inputs,"asset_id, station"
6,Missing Values Fixed,0
7,Duplicates Removed,0
8,Outliers Treated,0 (none removed or capped)
9,Train Rows,"4,900 (70.0%)"


Preprocessing complete. No machine learning model was trained.
